In [13]:
import time
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from torchvision import datasets, transforms
from sklearn.metrics import classification_report

In [14]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(DEVICE)

def train_on_batch(batch_x,
                   batch_y,
                   model,
                   loss_function,
                   optim):
    model.train()
    model.zero_grad()
    
    batch_x = batch_x.to(DEVICE)
    batch_y = batch_y.to(DEVICE)

    output = model(batch_x)
    loss = loss_function(output, batch_y)
    loss.backward()
    optim.step()
    
def train_epoch(batch_size,
                dataset,
                model,
                loss_function,
                optim):
    for batch_x, batch_y in torch.utils.data.DataLoader(dataset=dataset, 
                                                        batch_size=batch_size, 
                                                        shuffle=True):
        train_on_batch(batch_x, batch_y, model, loss_function, optim)

def trainer(epochs_amount,
            batch_size,
            dataset,
            model,
            loss_function,
            optimizer,
            lr=0.01):
    optim = optimizer(model.parameters(), lr=lr)
    for epoch in range(epochs_amount):
        start_time = time.time()
        train_epoch(batch_size, dataset, model, loss_function, optim)
        end_time = time.time()
        print(f'Эпоха {epoch+1} завершена за {round(end_time-start_time, 2)} секунд')

def tester(batch_size,
           dataset,
           model):
    y_predict, y_test = [], []
    for batch_x, batch_y in torch.utils.data.DataLoader(dataset=dataset, 
                                                        batch_size=batch_size):
        with torch.no_grad():
            model.eval()
            
            batch_x = batch_x.to(DEVICE)
            batch_y = batch_y.to(DEVICE)
    
            output = model(batch_x)
            y_predict.extend(torch.argmax(output, dim=1))
            y_test.extend(batch_y)
    print(classification_report(y_test, y_predict))

cpu


In [4]:
#Загрузка данных
train_data = datasets.MNIST('./mnist', train=True, download=True, transform=transforms.ToTensor())
test_data = datasets.MNIST('./mnist', train=False, download=True, transform=transforms.ToTensor())

In [5]:
#CNN
class CNN(torch.nn.Module):
    def __init__(self):
        super(CNN, self).__init__()

        self.layers = torch.nn.Sequential()
        self.layers.add_module('conv1', torch.nn.Conv2d(1,6,5))
        self.layers.add_module('pooling1', torch.nn.MaxPool2d(2))
        self.layers.add_module('relu1', torch.nn.ReLU())
        self.layers.add_module('conv2', torch.nn.Conv2d(6,36,5))
        self.layers.add_module('pooling2', torch.nn.MaxPool2d(2))
        self.layers.add_module('tanh2', torch.nn.Tanh())
        self.layers.add_module('flatten', torch.nn.Flatten())
        self.layers.add_module('lin3', torch.nn.Linear(576,80))
        self.layers.add_module('relu3', torch.nn.ReLU())
        self.layers.add_module('lin4', torch.nn.Linear(80,35))
        self.layers.add_module('relu4', torch.nn.ReLU())
        self.layers.add_module('classification', torch.nn.Linear(35,10))

    def forward(self, input):
        return self.layers(input)

In [6]:
#Создание модели
model = CNN()
model.to(DEVICE);

loss_func = torch.nn.CrossEntropyLoss()
optim_func = torch.optim.Adam

In [18]:
#Обучение моделей
trainer(10,
        128,
        train_data,
        model,
        loss_func,
        optim_func,
        0.01)

Эпоха 1 завершена за 22.64 секунд
Эпоха 2 завершена за 23.84 секунд
Эпоха 3 завершена за 21.89 секунд
Эпоха 4 завершена за 16.03 секунд
Эпоха 5 завершена за 15.59 секунд
Эпоха 6 завершена за 15.36 секунд
Эпоха 7 завершена за 15.27 секунд
Эпоха 8 завершена за 16.58 секунд
Эпоха 9 завершена за 11.69 секунд
Эпоха 10 завершена за 9.91 секунд


In [17]:
#Тестирование модели
tester(128,
       test_data,
       model)

              precision    recall  f1-score   support

           0       0.99      0.99      0.99       980
           1       0.99      1.00      0.99      1135
           2       0.98      0.98      0.98      1032
           3       0.98      0.99      0.99      1010
           4       0.99      0.98      0.99       982
           5       0.99      0.98      0.99       892
           6       0.99      0.99      0.99       958
           7       0.99      0.98      0.98      1028
           8       0.97      0.99      0.98       974
           9       0.97      0.97      0.97      1009

    accuracy                           0.99     10000
   macro avg       0.99      0.99      0.99     10000
weighted avg       0.99      0.99      0.99     10000

